In [ ]:
# ============ Paso 0 · ¿Qué permite el dato? Factibilidad del diseño por proceso ============
# Antes de escribir nada en el anteproyecto, mide tres cosas, sin modificar ningún archivo:
#   A. Poblaciones: cuántos procesos y cuántos positivos SRI quedan en cada familia, si en 2026
#      desaparecen menor cuantía y cotización, y qué hay dentro del catálogo electrónico.
#   B. Cobertura de los campos que necesitan los indicadores aprobados (precio por CPC,
#      oferentes, plazos) en el universo sin catálogo.
#   C. Qué trae el JSON crudo que la base no tiene: estado del proceso (desierto, cancelado),
#      fechas, contratos, modificaciones y ejecución. Decide si se puede validar con lo que
#      pasó DESPUÉS de la adjudicación. Lee una muestra de dos años: tarda unos minutos.
import os, time
import numpy as np, pandas as pd, duckdb

os.chdir(r'C:\SERCOP_FINAL')
R    = '03_resultados'
BASE = f'{R}/final_procesos.parquet'
con  = duckdb.connect()
con.sql("SET preserve_insertion_order=false")
con.sql(f"CREATE OR REPLACE VIEW fp AS SELECT * FROM read_parquet('{BASE}')")
C = [r[0] for r in con.sql("DESCRIBE fp").fetchall()]
def hay(*cs): return all(c in C for c in cs)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30)
N = con.sql("SELECT count(*) FROM fp").fetchone()[0]
print(f'base: {N:,} procesos x {len(C)} columnas')

NOCAT = "NOT coalesce(CAST(es_catalogo AS BOOLEAN), false)" if hay('es_catalogo') else "familia_procedimiento <> 'Catalogo electronico'"
POS   = "coalesce(CAST(y_fantasma AS INTEGER), 0) = 1" if hay('y_fantasma') else "false"

# ---------------- A1. Poblaciones y positivos por familia ----------------
print('\n' + '='*90 + '\nA1. Procesos, monto y positivos SRI por familia de procedimiento\n' + '='*90)
A1 = con.sql(f"""
SELECT familia_procedimiento AS familia, count(*) AS procesos,
       round(100.0*count(*)/{N}, 2) AS pct,
       round(sum(monto_final)/1e6, 1) AS monto_mm,
       CAST(sum(CASE WHEN {POS} THEN 1 ELSE 0 END) AS BIGINT) AS proc_positivos,
       count(DISTINCT CASE WHEN {POS} THEN proveedor_ruc END) AS ruc_positivos
FROM fp GROUP BY 1 ORDER BY procesos DESC""").df()
print(A1.to_string(index=False))
tot = con.sql(f"""SELECT CAST(sum(CASE WHEN {POS} THEN 1 ELSE 0 END) AS BIGINT),
                         count(DISTINCT CASE WHEN {POS} THEN proveedor_ruc END),
                         CAST(sum(CASE WHEN {POS} AND {NOCAT} THEN 1 ELSE 0 END) AS BIGINT),
                         count(DISTINCT CASE WHEN {POS} AND {NOCAT} THEN proveedor_ruc END),
                         CAST(sum(CASE WHEN {NOCAT} THEN 1 ELSE 0 END) AS BIGINT)
                  FROM fp""").fetchone()
print(f'\npositivos: {tot[0]:,} procesos de {tot[1]} RUC en total | sin catálogo: '
      f'{tot[2]:,} procesos de {tot[3]} RUC, dentro de {tot[4]:,} procesos')

# ---------------- A2. Familia x año (sin catálogo) ----------------
print('\n' + '='*90 + '\nA2. Procesos por familia y año, sin catálogo (¿desaparecen familias en 2026?)\n' + '='*90)
A2 = con.sql(f"""SELECT familia_procedimiento AS familia, anio_proceso AS anio, count(*) AS n
                 FROM fp WHERE {NOCAT} GROUP BY 1, 2""").df()
T2 = A2.pivot_table(index='familia', columns='anio', values='n', aggfunc='sum', fill_value=0)
print(T2.to_string())

# ---------------- A3. Qué hay dentro del catálogo ----------------
if hay('tipo_contratacion'):
    print('\n' + '='*90 + '\nA3. Las 12 descripciones más frecuentes dentro del catálogo electrónico\n' + '='*90)
    A3 = con.sql(f"""SELECT left(tipo_contratacion, 80) AS descripcion, count(*) AS procesos,
                            round(avg(monto_final), 0) AS monto_medio
                     FROM fp WHERE NOT ({NOCAT}) GROUP BY 1 ORDER BY 2 DESC LIMIT 12""").df()
    print(A3.to_string(index=False))

# ---------------- B. Cobertura de campos para los indicadores (sin catálogo) ----------------
print('\n' + '='*90 + '\nB1. % de procesos con dato, por familia (sin catálogo)\n' + '='*90)
campos = [c for c in ['monto_award', 'monto_tender', 'n_oferentes', 'proveedor_unico',
                      'cpc5_principal', 'fecha_fin_tender', 'fecha_proc'] if c in C]
sel = ', '.join(f"round(avg(CASE WHEN {c} IS NOT NULL THEN 100.0 ELSE 0 END), 1) AS {c}" for c in campos)
B1 = con.sql(f"""SELECT familia_procedimiento AS familia, count(*) AS procesos, {sel}
                 FROM fp WHERE {NOCAT} GROUP BY 1 ORDER BY procesos DESC""").df()
print(B1.to_string(index=False))

if hay('cpc5_principal'):
    print('\n' + '='*90 + '\nB2. ¿Alcanzan los grupos comparables para medir precio? (familia x CPC x año, >= 30 procesos)\n' + '='*90)
    B2 = con.sql(f"""
    WITH g5 AS (SELECT familia_procedimiento f, cpc5_principal c, anio_proceso a, count(*) n
                FROM fp WHERE {NOCAT} AND cpc5_principal IS NOT NULL GROUP BY 1,2,3),
         g3 AS (SELECT familia_procedimiento f, left(CAST(cpc5_principal AS VARCHAR), 3) c, anio_proceso a, count(*) n
                FROM fp WHERE {NOCAT} AND cpc5_principal IS NOT NULL GROUP BY 1,2,3)
    SELECT g5.f AS familia, CAST(sum(g5.n) AS BIGINT) AS con_cpc,
           round(100.0*sum(CASE WHEN g5.n >= 30 THEN g5.n ELSE 0 END)/sum(g5.n), 1) AS pct_en_grupo_cpc5,
           (SELECT round(100.0*sum(CASE WHEN g3.n >= 30 THEN g3.n ELSE 0 END)/sum(g3.n), 1)
              FROM g3 WHERE g3.f = g5.f) AS pct_en_grupo_cpc3
    FROM g5 GROUP BY 1 ORDER BY con_cpc DESC""").df()
    print(B2.to_string(index=False))

# ---------------- C. Lo que trae el JSON crudo (v3: lectura por streaming) ----------------
# Los archivos del SERCOP son PAQUETES OCDS: {"releases": [...]} o una lista de paquetes
# [{"releases": [...]}, ...]. Por eso fallaba la lectura "un release por línea" de la v1/v2.
# Aquí se leen release por release con ijson, igual que en la celda 6 del cuaderno 6: no carga
# el archivo de 1,7 GB en memoria. Además mide si el plazo o el monto del contrato CAMBIAN entre
# releases del mismo proceso: el tercer patrón del art. 21 de la LOSNCP.
try:
    import ijson
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ijson']); import ijson
from collections import Counter

ANIOS  = [2019, 2024]   # dos años de regímenes distintos; puedes cambiarlos
LIMITE = None           # releases por año (None = el año completo, alrededor de 1 a 2 minutos por año)

def archivo(a):
    rep = f'00_json_crudo/releases_{a}_reparado.json'
    return rep if os.path.exists(rep) else f'00_json_crudo/releases_{a}.json'

def prefijo_de(ruta):
    """Lista de paquetes -> 'item.releases.item'; un solo paquete -> 'releases.item'."""
    with open(ruta, 'rb') as fh:
        while True:
            c = fh.read(1)
            if not c: return 'releases.item'
            if not c.isspace():
                return 'item.releases.item' if c == b'[' else 'releases.item'

def primero(v):
    """Primer elemento de una lista; tolera la mala serialización {'0': ..., '1': ...}."""
    if isinstance(v, list): v = v[0] if v else None
    elif isinstance(v, dict) and v and all(str(k).isdigit() for k in v): v = v[sorted(v, key=int)[0]]
    else: v = None
    return v if isinstance(v, dict) else {}

def largo(v):
    if isinstance(v, list): return len(v)
    if isinstance(v, dict) and v and all(str(k).isdigit() for k in v): return len(v)
    return 0

def g(d, *ruta):
    for k in ruta:
        if not isinstance(d, dict): return None
        d = d.get(k)
    return d

def num(v):
    try: return float(v)
    except (TypeError, ValueError): return None

def fila(rel):
    t  = rel.get('tender') if isinstance(rel.get('tender'), dict) else {}
    aw = primero(rel.get('awards')); ct = primero(rel.get('contracts')); it = primero(t.get('items'))
    return {
        'ocid': rel.get('ocid'),
        'm': str(t.get('procurementMethodDetails') or '').upper(),
        't_status': t.get('status'),
        'fecha_inicio_ofertas': g(t, 'tenderPeriod', 'startDate') is not None,
        'fecha_cierre_ofertas': g(t, 'tenderPeriod', 'endDate') is not None,
        'fecha_publicacion': t.get('datePublished') is not None,
        'fecha_preguntas': g(t, 'enquiryPeriod', 'endDate') is not None,
        'periodo_adjudicacion': (g(t, 'awardPeriod', 'startDate') is not None or
                                 g(t, 'awardPeriod', 'endDate') is not None),
        'fecha_adjudicacion': aw.get('date') is not None,
        'monto_adj': num(g(aw, 'value', 'amount')),
        'hay_contrato': largo(rel.get('contracts')) > 0,
        'c_status': ct.get('status'),
        'fecha_firma': ct.get('dateSigned') is not None,
        'monto_contrato': num(g(ct, 'value', 'amount')),
        'plazo_contrato': g(ct, 'period', 'endDate') is not None,
        'c_fin_txt': g(ct, 'period', 'endDate'),
        't_fin_txt': g(t, 'tenderPeriod', 'endDate'),
        'enmiendas_pliego': largo(t.get('amendments')) > 0,
        'modificaciones': largo(ct.get('amendments')) > 0,
        'ejecucion': ct.get('implementation') is not None,
        'pagos': largo(g(ct, 'implementation', 'transactions')) > 0,
        'cpc_item': g(it, 'classification', 'id') is not None,
        'cantidad_item': it.get('quantity') is not None,
        'precio_unitario': g(it, 'unit', 'value', 'amount') is not None,
        'oferentes_identificados': largo(t.get('tenderers')) > 0,
        'bids': rel.get('bids') is not None,
    }

FAM = """CASE WHEN m LIKE '%CATALOGO%' OR m LIKE '%CATÁLOGO%' THEN 'Catalogo electronico'
              WHEN m LIKE '%SUBASTA INVERSA%' THEN 'Subasta inversa electronica'
              WHEN m LIKE '%LICITACION%' OR m LIKE '%LICITACIÓN%' THEN 'Licitacion'
              WHEN m LIKE '%COTIZACION%' OR m LIKE '%COTIZACIÓN%' THEN 'Cotizacion'
              WHEN m LIKE '%MENOR CUANTIA%' OR m LIKE '%MENOR CUANTÍA%' THEN 'Menor cuantia'
              WHEN m LIKE '%LISTA CORTA%' THEN 'Lista corta'
              WHEN m LIKE '%CONCURSO PUBLICO%' OR m LIKE '%CONCURSO PÚBLICO%' THEN 'Concurso publico'
              WHEN m LIKE '%CONTRATACION DIRECTA%' OR m LIKE '%CONTRATACIÓN DIRECTA%' THEN 'Contratacion directa'
              WHEN m LIKE '%REGIMEN ESPECIAL%' OR m LIKE '%RÉGIMEN ESPECIAL%' THEN 'Regimen especial'
              WHEN m = '' THEN '(sin metodo: solo planeacion)'
              ELSE 'Otras' END"""
BOOLS = ['fecha_publicacion', 'fecha_inicio_ofertas', 'fecha_cierre_ofertas', 'fecha_preguntas',
         'periodo_adjudicacion', 'fecha_adjudicacion', 'hay_contrato', 'fecha_firma', 'plazo_contrato',
         'modificaciones', 'ejecucion', 'pagos', 'cpc_item', 'cantidad_item', 'precio_unitario',
         'oferentes_identificados', 'bids', 'enmiendas_pliego']

RES = []
for a in ANIOS:
    ruta = archivo(a)
    if not os.path.exists(ruta):
        print(f'\nno existe {ruta}'); continue
    t0, filas_r, tags = time.time(), [], Counter()
    with open(ruta, 'rb') as fh:
        for k, rel in enumerate(ijson.items(fh, prefijo_de(ruta))):
            if LIMITE and k >= LIMITE: break
            if not isinstance(rel, dict): continue
            tg = rel.get('tag')
            tags.update(tg if isinstance(tg, list) else [str(tg)])
            filas_r.append(fila(rel))
            if (k + 1) % 100_000 == 0:
                print(f'    {k+1:,} releases ({time.time()-t0:.0f} s)', end='\r')
    X = pd.DataFrame(filas_r); del filas_r
    print('\n' + '='*90 + f'\nC. {ruta}: {len(X):,} releases leídos ({time.time()-t0:.0f} s)\n' + '='*90)
    if X.empty:
        print('  no se leyó ningún release: pega el error o las primeras líneas del archivo'); continue
    print('etapas publicadas (tag):')
    print(pd.Series(tags, name='releases').sort_values(ascending=False).to_string())

    con.register('x_df', X)
    con.sql("CREATE OR REPLACE TEMP TABLE x AS SELECT * FROM x_df")
    con.unregister('x_df')
    agg = ', '.join(f'bool_or({b}) AS {b}' for b in BOOLS)
    con.sql(f"""CREATE OR REPLACE TEMP TABLE p AS
      SELECT ocid, {FAM.replace('m LIKE', 'max(m) LIKE').replace('m =', 'max(m) =')} AS familia,
             bool_or(t_status = 'unsuccessful') AS desierto,
             bool_or(t_status = 'cancelled')    AS cancelado,
             bool_or(c_status = 'terminated')   AS contrato_terminado,
             bool_or(c_status = 'cancelled')    AS contrato_cancelado,
             max(monto_adj) AS monto_adj, max(monto_contrato) AS monto_contrato,
             count(DISTINCT c_fin_txt) > 1      AS plazo_contrato_cambia,
             count(DISTINCT monto_contrato) > 1 AS monto_contrato_cambia,
             count(DISTINCT t_fin_txt) > 1      AS cierre_ofertas_cambia, {agg}
      FROM x GROUP BY ocid""")
    cols_pct = (['desierto', 'cancelado', 'contrato_terminado', 'contrato_cancelado',
                 'plazo_contrato_cambia', 'monto_contrato_cambia', 'cierre_ofertas_cambia'] + BOOLS)
    selp = ', '.join(f'round(100.0*avg(CASE WHEN {b} THEN 1 ELSE 0 END), 1) AS {b}' for b in cols_pct)
    P = con.sql(f"""SELECT familia, count(*) AS procesos, {selp},
                    round(100.0*avg(CASE WHEN monto_adj > 0 AND monto_contrato > 0
                                          AND abs(monto_contrato/monto_adj - 1) > 0.10 THEN 1 ELSE 0 END), 1)
                      AS contrato_difiere_10pct
                    FROM p GROUP BY 1 ORDER BY procesos DESC""").df()
    print('\n% de procesos (ocid) con cada dato, por familia:')
    print(P.set_index('familia').T.to_string())
    for col, txt in (('t_status', 'estado del proceso (tender.status)'), ('c_status', 'estado del contrato')):
        d = con.sql(f"SELECT {col} AS estado, count(DISTINCT ocid) AS procesos FROM x GROUP BY 1 ORDER BY 2 DESC").df()
        print(f'\n{txt}:'); print(d.to_string(index=False))
    P.insert(0, 'anio', a); RES.append(P)

if RES:
    pd.concat(RES).to_csv(f'{R}/c14_factibilidad_json.csv', index=False)
    print(f'\n  -> {R}/c14_factibilidad_json.csv')
print('\nPega aquí toda la salida de la parte C.')

In [ ]:
# ============ Inventario de bases de datos en tu computadora (solo lectura) ============
# Busca archivos de datos en la carpeta del proyecto y en Descargas, Documentos y Escritorio.
# No abre ni modifica nada: solo anota nombre, carpeta, tamaño y fecha. Fuera del proyecto
# muestra solo los archivos cuyo nombre sugiere que sirven para la tesis (SRI, Contraloría, etc.).
import os, time
import pandas as pd

EXT = ('.parquet', '.csv', '.xlsx', '.xls', '.json', '.zip', '.rar', '.7z', '.db', '.sqlite', '.accdb')
CLAVES = ('sri', 'catastro', 'fantasma', 'inexist', 'contralor', 'incumpl', 'fallid', 'supercia',
          'compan', 'sercop', 'ocds', 'release', 'rup', 'proveedor', 'contrat', 'compra',
          'beneficiari', 'ruc', 'directorio', 'sancion')
PROYECTO = r'C:\SERCOP_FINAL'
casa = os.path.expanduser('~')
RAICES = [PROYECTO] + [os.path.join(casa, *p) for p in
          [('Downloads',), ('Descargas',), ('Documents',), ('Documentos',), ('Desktop',), ('Escritorio',),
           ('OneDrive', 'Documentos'), ('OneDrive', 'Documents'), ('OneDrive', 'Escritorio'),
           ('OneDrive', 'Desktop')]]
SALTAR = {'node_modules', '.git', '__pycache__', '.ipynb_checkpoints', 'AppData', '$RECYCLE.BIN'}

filas, vistos = [], set()
for raiz in RAICES:
    if not os.path.isdir(raiz): continue
    for d, subdirs, archivos in os.walk(raiz):
        subdirs[:] = [s for s in subdirs if s not in SALTAR]
        for f in archivos:
            if not f.lower().endswith(EXT): continue
            ruta = os.path.normcase(os.path.abspath(os.path.join(d, f)))
            if ruta in vistos: continue
            vistos.add(ruta)
            try: st = os.stat(ruta)
            except OSError: continue
            filas.append({'raiz': raiz, 'carpeta': os.path.relpath(d, raiz), 'archivo': f,
                          'mb': round(st.st_size / 1e6, 1),
                          'modificado': time.strftime('%Y-%m-%d', time.localtime(st.st_mtime))})
INV = pd.DataFrame(filas)
if INV.empty:
    print('no se encontraron archivos de datos en esas carpetas')
else:
    INV['en_proyecto'] = INV.raiz.eq(PROYECTO)
    INV['relevante'] = INV.archivo.str.lower().str.contains('|'.join(CLAVES))
    print('Resumen por carpeta raíz:')
    print(INV.groupby('raiz').agg(archivos=('archivo', 'size'),
                                  gb=('mb', lambda s: round(s.sum() / 1024, 2))).to_string())
    P = INV[INV.en_proyecto]
    if len(P):
        print('\nProyecto, por subcarpeta de primer nivel:')
        P = P.assign(sub=P.carpeta.str.split(r'[\\/]').str[0])
        print(P.groupby('sub').agg(archivos=('archivo', 'size'), mb=('mb', 'sum'),
                                   ultimo=('modificado', 'max')).sort_values('mb', ascending=False).to_string())
        print('\nProyecto, archivos que no son c8_/c9_/final_ y pesan más de 1 MB (fuentes externas):')
        ext = P[~P.archivo.str.startswith(('c8_', 'c9_', 'c14_', 'final_')) & (P.mb > 1)]
        print(ext.sort_values('mb', ascending=False)[['carpeta', 'archivo', 'mb', 'modificado']]
                 .head(60).to_string(index=False))
    F = INV[~INV.en_proyecto & INV.relevante]
    print(f'\nFuera del proyecto: {len(F)} archivos con nombre relacionado con la tesis')
    if len(F):
        print(F.sort_values('modificado', ascending=False)[['raiz', 'carpeta', 'archivo', 'mb', 'modificado']]
                .head(60).to_string(index=False))
    os.makedirs(os.path.join(PROYECTO, '03_resultados'), exist_ok=True)
    salida = os.path.join(PROYECTO, '03_resultados', 'c14_inventario_compu.csv')
    INV.to_csv(salida, index=False)
    print(f'\n  -> {salida}  ({len(INV)} archivos de datos)')

In [ ]:
# ============ Verificación · releases por archivo y datos de ejecución en cualquier contrato ============
# Recorre los 12 archivos (unos 8 a 10 minutos). Cuenta releases y procesos distintos, y busca
# ejecución, modificaciones y ofertas en CUALQUIER contrato o adjudicación, no solo en el primero.
import os, glob, time
import pandas as pd
import ijson

def prefijo_de(ruta):
    with open(ruta, 'rb') as fh:
        while True:
            c = fh.read(1)
            if not c: return 'releases.item'
            if not c.isspace():
                return 'item.releases.item' if c == b'[' else 'releases.item'

def lista(v):
    if isinstance(v, list): return [x for x in v if isinstance(x, dict)]
    if isinstance(v, dict) and v and all(str(k).isdigit() for k in v):
        return [x for x in v.values() if isinstance(x, dict)]
    return []

filas = []
for ruta in sorted(glob.glob('00_json_crudo/releases_*.json')):
    if ruta.endswith('releases_2024.json') and os.path.exists(ruta.replace('.json', '_reparado.json')):
        continue                                    # para 2024 se usa el reparado
    t0, n, ocids = time.time(), 0, set()
    c = dict(con_implementacion=0, con_transacciones=0, con_hitos=0, con_modif_contrato=0,
             con_modif_adjudicacion=0, con_modif_pliego=0, con_bids=0, varios_contratos=0)
    with open(ruta, 'rb') as fh:
        for rel in ijson.items(fh, prefijo_de(ruta)):
            if not isinstance(rel, dict): continue
            n += 1; ocids.add(rel.get('ocid'))
            cts = lista(rel.get('contracts')); aws = lista(rel.get('awards'))
            t = rel.get('tender') if isinstance(rel.get('tender'), dict) else {}
            imps = [x.get('implementation') for x in cts if isinstance(x.get('implementation'), dict)]
            c['con_implementacion']     += bool(imps)
            c['con_transacciones']      += any(lista(i.get('transactions')) for i in imps)
            c['con_hitos']              += any(lista(i.get('milestones')) for i in imps)
            c['con_modif_contrato']     += any(lista(x.get('amendments')) for x in cts)
            c['con_modif_adjudicacion'] += any(lista(x.get('amendments')) for x in aws)
            c['con_modif_pliego']       += bool(lista(t.get('amendments')))
            c['con_bids']               += rel.get('bids') is not None
            c['varios_contratos']       += len(cts) > 1
    filas.append({'archivo': os.path.basename(ruta), 'releases': n, 'procesos': len(ocids),
                  'releases_por_proceso': round(n / max(len(ocids), 1), 2), **c,
                  'segundos': round(time.time() - t0)})
    print(f"  {os.path.basename(ruta)}: {n:,} releases, {len(ocids):,} procesos ({time.time()-t0:.0f} s)")

V = pd.DataFrame(filas)
print('\n' + V.to_string(index=False))
print(f"\nTOTAL: {V.releases.sum():,} releases | {V.procesos.sum():,} procesos")
V.to_csv('03_resultados/c14_verificacion_releases.csv', index=False)
print('  -> 03_resultados/c14_verificacion_releases.csv')